# Soft Cube Goal-Pose RL — Non-Warp

Goal: move an 8-node deformable cube on a table to a specific **position and orientation**.

Default target:
- COM position: `(0.55, 0.25, SIDE/2)`
- orientation: `+60°` yaw

Because a soft body is deformable, orientation is defined as the **Kabsch best-fit rigid rotation**
from the reference cube vertices to the current vertices.

Physics:
- 8 masses
- 28 spring-damper connections: 12 edges + 12 face diagonals + 4 body diagonals
- 12 actuated edge springs
- strain-based spring option similar to `real2sim-eval`
- axial dashpot + global drag
- table contact using restitution + Coulomb-like tangential impulse

In [ ]:
%pip -q install numpy matplotlib gymnasium torch

In [ ]:
import numpy as np, math
SIDE = 0.40

def cube_topology(side=SIDE, z0=0.22):
    X = np.array([
        [-1,-1,-1],[ 1,-1,-1],[-1, 1,-1],[ 1, 1,-1],
        [-1,-1, 1],[ 1,-1, 1],[-1, 1, 1],[ 1, 1, 1],
    ], dtype=np.float32)*(side/2)
    X[:,2] += z0
    springs, act_id, edge_count = [], [], 0
    for i in range(8):
        for j in range(i+1,8):
            springs.append((i,j))
            L = np.linalg.norm(X[j]-X[i])
            if np.isclose(L, side, atol=1e-5):
                act_id.append(edge_count); edge_count += 1
            else:
                act_id.append(-1)
    si=np.array([i for i,j in springs],np.int32)
    sj=np.array([j for i,j in springs],np.int32)
    L0=np.linalg.norm(X[sj]-X[si],axis=1).astype(np.float32)
    return X,si,sj,L0,np.array(act_id,np.int32)

X_REF, SI, SJ, L0, ACT_ID = cube_topology()
N_ACT=int(np.sum(ACT_ID>=0))
REF_CENTERED=X_REF-X_REF.mean(axis=0,keepdims=True)

def rotz(yaw):
    c,s=np.cos(yaw),np.sin(yaw)
    return np.array([[c,-s,0],[s,c,0],[0,0,1]],np.float32)

def estimate_rotation_kabsch(x):
    A=REF_CENTERED
    B=x-x.mean(axis=0,keepdims=True)
    H=A.T@B
    U,_,Vt=np.linalg.svd(H)
    R=Vt.T@U.T
    if np.linalg.det(R)<0:
        Vt[-1,:]*=-1
        R=Vt.T@U.T
    return R.astype(np.float32)

def rotation_angle(Ra,Rb):
    R=Ra.T@Rb
    c=np.clip((np.trace(R)-1.0)/2.0,-1.0,1.0)
    return float(np.arccos(c))

print("particles",len(X_REF),"springs",len(SI),"actuators",N_ACT)

In [ ]:
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces

GOAL_POS=np.array([0.55,0.25,SIDE/2],np.float32)
GOAL_R=rotz(np.deg2rad(60.0))

## Pose objective

\[
d_p=\|p-p_g\|,
\qquad
d_R=\cos^{-1}\left(\frac{\operatorname{tr}(R_g^TR)-1}{2}\right)
\]

Reward uses progress in both errors, plus action/deformation penalties.

In [ ]:
class SoftCubeSim:
    def __init__(self, dt=0.0025, substeps=4, mass=0.15,
                 spring_mode="strain", stiffness=72.0, dashpot=2.0,
                 drag=0.5, act_amp=0.20, restitution=0.15, friction=0.65):
        self.dt=dt; self.substeps=substeps; self.mass=mass
        self.spring_mode=spring_mode; self.stiffness=stiffness
        self.dashpot=dashpot; self.drag=drag; self.act_amp=act_amp
        self.restitution=restitution; self.friction=friction
        self.reset()

    def reset(self, rng=None):
        rng=np.random.default_rng() if rng is None else rng
        self.x=X_REF.copy()+rng.normal(0,0.002,X_REF.shape).astype(np.float32)
        self.v=np.zeros_like(self.x)
        return self.x.copy()

    def pose(self):
        return self.x.mean(axis=0), estimate_rotation_kabsch(self.x)

    def _spring_forces(self, action):
        F=np.zeros_like(self.x)
        for s,(i,j) in enumerate(zip(SI,SJ)):
            d=self.x[j]-self.x[i]
            L=np.linalg.norm(d)+1e-8
            n=d/L
            target=L0[s]
            aid=ACT_ID[s]
            if aid>=0:
                target*=1.0+self.act_amp*float(action[aid])

            if self.spring_mode=="strain":
                smag=self.stiffness*(L/target-1.0)
            else:
                smag=self.stiffness*(L-target)

            rel=float(np.dot(self.v[j]-self.v[i],n))
            f=(smag+self.dashpot*rel)*n
            F[i]+=f; F[j]-=f
        return F

    def _table_contact(self, x0, v0):
        dt=self.dt
        x1=x0.copy(); v1=v0.copy()
        for p in range(8):
            z=float(x0[p,2]); vz=float(v0[p,2])
            znext=z+vz*dt
            if znext<0.0 and vz<-1e-6:
                vn=np.array([0,0,vz],np.float32)
                vt=np.array([v0[p,0],v0[p,1],0],np.float32)
                vn_len=abs(vz)
                vt_len=np.linalg.norm(vt)+1e-8

                vn_new=-self.restitution*vn
                scale=max(0.0,1.0-self.friction*(1.0+self.restitution)*vn_len/vt_len)
                vc=vn_new+scale*vt

                toi=np.clip(-z/vz,0.0,dt) if z>0 else 0.0
                x1[p]=x0[p]+v0[p]*toi+vc*(dt-toi)
                x1[p,2]=max(0.0,x1[p,2])
                v1[p]=vc
            else:
                x1[p]=x0[p]+v0[p]*dt
                if x1[p,2]<0:
                    x1[p,2]=0
                    if v1[p,2]<0: v1[p,2]=0
        return x1,v1

    def step(self, action):
        action=np.clip(np.asarray(action,np.float32),-1,1)
        for _ in range(self.substeps):
            F=self._spring_forces(action)
            F[:,2]-=self.mass*9.81
            self.v += (F/self.mass)*self.dt
            self.v *= np.exp(-self.drag*self.dt)
            self.x,self.v=self._table_contact(self.x,self.v)
        return self.x.copy(),self.v.copy()

In [ ]:
sim=SoftCubeSim()
for _ in range(600):
    sim.step(np.zeros(N_ACT,np.float32))
p,R=sim.pose()
print("passive COM:",p)
print("min z:",sim.x[:,2].min())
print("orientation drift deg:",np.rad2deg(rotation_angle(np.eye(3),R)))

In [ ]:
class GoalPoseCubeEnv(gym.Env):
    def __init__(self, episode_steps=600):
        super().__init__()
        self.sim=SoftCubeSim()
        self.episode_steps=episode_steps
        self.goal_pos=GOAL_POS.copy()
        self.goal_R=GOAL_R.copy()
        self.action_space=spaces.Box(-1,1,shape=(N_ACT,),dtype=np.float32)
        # relative vertices 24 + velocities 24 + goal delta 3 + rotation error matrix 9
        self.observation_space=spaces.Box(-np.inf,np.inf,shape=(60,),dtype=np.float32)

    def _errors(self):
        p,R=self.sim.pose()
        return float(np.linalg.norm(self.goal_pos-p)), rotation_angle(R,self.goal_R), p, R

    def _obs(self):
        p,R=self.sim.pose()
        rel=self.sim.x-p[None,:]
        goal_delta=self.goal_pos-p
        Rerr=self.goal_R.T@R
        return np.concatenate([rel.reshape(-1),self.sim.v.reshape(-1),
                               goal_delta,Rerr.reshape(-1)]).astype(np.float32)

    def reset(self,seed=None,options=None):
        super().reset(seed=seed)
        self.sim.reset(self.np_random)
        self.t=0; self.success_count=0
        self.prev_dp,self.prev_da,_,_=self._errors()
        return self._obs(),{"pos_error":self.prev_dp,"angle_error":self.prev_da}

    def step(self,action):
        self.sim.step(action); self.t+=1
        dp,da,p,R=self._errors()
        curL=np.linalg.norm(self.sim.x[SJ]-self.sim.x[SI],axis=1)
        deform=np.mean(((curL-L0)/L0)**2)

        reward=(5.0*(self.prev_dp-dp)+1.0*(self.prev_da-da)
                -0.002*np.mean(np.square(action))-0.01*deform)

        ok=(dp<0.05) and (da<np.deg2rad(15))
        self.success_count=self.success_count+1 if ok else 0
        success=self.success_count>=20
        reward += 0.02*float(ok) + 5.0*float(success)

        failed=(not np.isfinite(self.sim.x).all()) or (np.abs(self.sim.x).max()>10)
        terminated=bool(success or failed)
        truncated=bool(self.t>=self.episode_steps)
        self.prev_dp,self.prev_da=dp,da
        info={"pos_error":dp,"angle_error":da,"angle_error_deg":np.rad2deg(da),
              "com":p.copy(),"success":success}
        return self._obs(),float(reward),terminated,truncated,info

env=GoalPoseCubeEnv()
obs,info=env.reset(seed=0)
print("obs",obs.shape,"action",env.action_space.shape,info)

## PPO

In [ ]:
import torch, torch.nn as nn
from torch.distributions import Normal
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ActorCritic(nn.Module):
    def __init__(self,obs_dim=60,act_dim=N_ACT,h=128):
        super().__init__()
        self.actor=nn.Sequential(nn.Linear(obs_dim,h),nn.Tanh(),nn.Linear(h,h),nn.Tanh(),nn.Linear(h,act_dim))
        self.critic=nn.Sequential(nn.Linear(obs_dim,h),nn.Tanh(),nn.Linear(h,h),nn.Tanh(),nn.Linear(h,1))
        self.log_std=nn.Parameter(torch.full((act_dim,),-0.5))
    def dist(self,o):
        mu=self.actor(o); return Normal(mu,self.log_std.exp().expand_as(mu))
    def value(self,o): return self.critic(o).squeeze(-1)
    def act(self,o):
        d=self.dist(o); z=d.rsample(); a=torch.tanh(z)
        lp=d.log_prob(z).sum(-1)-torch.log(1-a*a+1e-6).sum(-1)
        return a,z,lp,self.value(o)
    def evaluate(self,o,z):
        d=self.dist(o); a=torch.tanh(z)
        lp=d.log_prob(z).sum(-1)-torch.log(1-a*a+1e-6).sum(-1)
        return lp,d.entropy().sum(-1),self.value(o)

def compute_gae(r,d,v,last_v,gamma=.99,lam=.95):
    A=np.zeros(len(r),np.float32); g=0.0
    for t in reversed(range(len(r))):
        nv=last_v if t==len(r)-1 else v[t+1]
        nt=1.0-d[t]
        delta=r[t]+gamma*nv*nt-v[t]
        g=delta+gamma*lam*nt*g
        A[t]=g
    return A,A+v

def train_ppo(updates=30,rollout_steps=1024,epochs=6,minibatch=256):
    env=GoalPoseCubeEnv(); net=ActorCritic().to(DEVICE)
    opt=torch.optim.Adam(net.parameters(),lr=3e-4)
    obs,_=env.reset(seed=0)
    hist={"return":[],"pos_error":[],"angle_deg":[]}; ep_ret=0.0

    for upd in range(updates):
        O=[];Z=[];LP=[];RR=[];DD=[];VV=[]
        for _ in range(rollout_steps):
            ot=torch.tensor(obs,dtype=torch.float32,device=DEVICE)[None]
            with torch.no_grad(): a,z,lp,val=net.act(ot)
            nxt,r,term,trunc,info=env.step(a[0].cpu().numpy())
            done=term or trunc
            O.append(obs.copy());Z.append(z[0].cpu().numpy());LP.append(lp.item())
            RR.append(r);DD.append(float(done));VV.append(val.item())
            ep_ret+=r; obs=nxt
            if done:
                hist["return"].append(ep_ret); hist["pos_error"].append(info["pos_error"])
                hist["angle_deg"].append(info["angle_error_deg"])
                ep_ret=0.0; obs,_=env.reset()

        with torch.no_grad():
            last_v=net.value(torch.tensor(obs,dtype=torch.float32,device=DEVICE)[None]).item()
        A,RET=compute_gae(np.array(RR,np.float32),np.array(DD,np.float32),np.array(VV,np.float32),last_v)
        A=(A-A.mean())/(A.std()+1e-8)

        O=torch.tensor(np.array(O),dtype=torch.float32,device=DEVICE)
        Z=torch.tensor(np.array(Z),dtype=torch.float32,device=DEVICE)
        OLDLP=torch.tensor(LP,dtype=torch.float32,device=DEVICE)
        A=torch.tensor(A,dtype=torch.float32,device=DEVICE)
        RET=torch.tensor(RET,dtype=torch.float32,device=DEVICE)

        n=len(RR)
        for _ in range(epochs):
            perm=torch.randperm(n,device=DEVICE)
            for st in range(0,n,minibatch):
                b=perm[st:st+minibatch]
                lp,ent,val=net.evaluate(O[b],Z[b])
                ratio=(lp-OLDLP[b]).exp()
                ploss=-torch.min(ratio*A[b],torch.clamp(ratio,.8,1.2)*A[b]).mean()
                loss=ploss+.5*(val-RET[b]).square().mean()-.002*ent.mean()
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(),.5); opt.step()
        if (upd+1)%5==0:
            recent=hist["return"][-5:]
            print(upd+1,np.mean(recent) if recent else np.nan)
    return net,hist

# smoke test:
# policy,hist=train_ppo(updates=3,rollout_steps=256)

# longer:
# policy,hist=train_ppo(updates=80,rollout_steps=1024)

## Experiments
- position-only vs position+orientation reward
- `spring_mode="hooke"` vs `"strain"`
- dashpot only vs dashpot + global drag
- sweep restitution/friction
- randomize goal pose at reset for general goal-conditioned control